In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# --- 1. 데이터 불러오기 ---
# ❗️ 사용자 수정 필요: 실제 엑셀 파일 경로로 변경하세요.
excel_path = 'C:\workspace\키움뱅크\ML\dart_credit_risk_result.xlsx'
try:
    df = pd.read_excel(excel_path)
except FileNotFoundError:
    print(f"오류: '{excel_path}' 파일을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    exit()

print("--- 데이터 일부 확인 ---")
print(df.head())
print("\n--- 데이터 정보 ---")
df.info()


# --- 2. 데이터 전처리 ---
# ❗️ 사용자 수정 필요: 실제 타겟 변수(예측하려는 값)의 열 이름으로 변경하세요.
target_column = 'Grade'

# 'corp_code'는 회사를 식별하는 고유 ID일 가능성이 높으므로, 모델 학습에서 제외합니다.
# 만약 중요한 feature라면 이 줄을 주석 처리하세요.
if 'corp_code' in df.columns:
    df = df.drop('corp_code', axis=1)

# X (독립 변수, Features)와 y (종속 변수, Target) 분리
X = df.drop(target_column, axis=1)
y = df[target_column]

# 범주형 데이터(문자열)를 숫자로 변환 (One-Hot Encoding)
X_encoded = pd.get_dummies(X, drop_first=True)

# 타겟 변수(신용등급)도 숫자(0, 1, 2...)로 변환 (Label Encoding)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 인코딩 후의 열 이름을 저장 (새로운 데이터 예측 시 사용)
encoded_columns = X_encoded.columns

print("\n--- 원-핫 인코딩 후 데이터 일부 확인 ---")
print(X_encoded.head())


# --- 3. 학습 및 테스트 데이터 분리 ---
# 데이터를 80%는 학습용, 20%는 검증용으로 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"\n학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")


# --- 4. XGBoost 모델 생성 및 학습 ---
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False
)

print("\n--- 모델 학습 시작 ---")
model.fit(X_train, y_train)
print("--- 모델 학습 완료 ---")


# --- 5. 모델 성능 평가 ---
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\n모델 예측 정확도: {accuracy * 100:.2f}%")

report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
print("\n--- 상세 평가 리포트 ---")
print(report)


# --- 6. 새로운 데이터 예측 예시 ---
# ❗️ 수정된 부분 1: 예측할 데이터에는 정답('Grade' 또는 '신용등급')을 포함하지 않습니다.
# ❗️ 수정된 부분 2: 'AT'와 'PD' 사이에 쉼표를 추가했습니다.
new_company_data = pd.DataFrame([{
    # 'corp_code'는 학습에서 제외했으므로 여기에서도 제외합니다.
    'CR': 1.74207198003175,
    'DER': 1.14099257484873,
    'ICR': 0,
    'OPM': 0.0407584511351937,
    'AT': 0.245063558668831, # 이 줄 끝에 쉼표가 없어서 추가했습니다.
    'PD': 0.175351018024796
    # 'Grade' 열은 예측 대상이므로 제거했습니다.
}])

# 새로운 데이터도 학습 데이터와 동일하게 인코딩
new_company_encoded = pd.get_dummies(new_company_data)
# 학습 시 사용된 열을 기준으로 재구성하여 열의 순서와 개수를 맞춥니다.
new_company_reindexed = new_company_encoded.reindex(columns=encoded_columns, fill_value=0)

# 예측 수행
prediction_encoded = model.predict(new_company_reindexed)

# 예측 결과(숫자)를 원래의 신용등급(문자)으로 변환
prediction_original = label_encoder.inverse_transform(prediction_encoded)

print(f"\n--- 새로운 기업 데이터 예측 결과 ---")
print(f"예측된 신용등급: {prediction_original[0]}")


--- 데이터 일부 확인 ---
   corp_code        CR       DER  ICR       OPM        AT        PD Grade
0     100601  1.075677  2.107564    0  0.037111  0.240051  0.094478     B
1     100939  1.907837  0.412088    0  0.053074  0.177101  0.000072    A0
2     101044  2.763931  0.283168    0 -0.474904  0.038977  0.000239    A0
3     101220  1.352805  1.140993    0  0.060384  0.315300  0.332609    CC
4     101257  1.742072  1.651780    0  0.001716  0.228900  0.081990     B

--- 데이터 정보 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2384 entries, 0 to 2383
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   corp_code  2384 non-null   int64  
 1   CR         2384 non-null   float64
 2   DER        2384 non-null   float64
 3   ICR        2384 non-null   int64  
 4   OPM        2384 non-null   float64
 5   AT         2384 non-null   float64
 6   PD         2384 non-null   float64
 7   Grade      2384 non-null   object 
dtypes: float64(

c:\Users\edukd\miniconda3\envs\kiwoombank\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:12:34] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [1]:
import pandas as pd

# 1. 방금 출력된 로그를 기반으로 records 리스트를 만듭니다.
# (np.float64는 pandas가 자동으로 처리하므로 숫자로 변경했습니다)
records = [{'company': 'CJ', 'news_sentiment_score': np.float64(-52.92), 'news_count': 30, 'sentiment_volatility': 0.4649, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9921},
    {'company': 'CJ ENM', 'news_sentiment_score': np.float64(-42.92), 'news_count': 30, 'sentiment_volatility': 0.5189, 'positive_ratio': 0.0333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.8867},
    {'company': 'CJ제일제당', 'news_sentiment_score': np.float64(-13.13), 'news_count': 30, 'sentiment_volatility': 0.5684, 'positive_ratio': 0.1333, 'negative_ratio': 0.2667, 'recency_weight_mean': 0.9185},
    {'company': 'HMM', 'news_sentiment_score': np.float64(-76.41), 'news_count': 30, 'sentiment_volatility': 0.5532, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9906},
    {'company': 'KT', 'news_sentiment_score': np.float64(-38.93), 'news_count': 29, 'sentiment_volatility': 0.826, 'positive_ratio': 0.2414, 'negative_ratio': 0.6207, 'recency_weight_mean': 0.987},
    {'company': 'KT&G', 'news_sentiment_score': np.float64(-46.87), 'news_count': 30, 'sentiment_volatility': 0.552, 'positive_ratio': 0.0333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.9173},
    {'company': 'LG생활건강', 'news_sentiment_score': np.float64(-74.43), 'news_count': 30, 'sentiment_volatility': 0.5427, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9612},
    {'company': 'LG에너지솔루션', 'news_sentiment_score': np.float64(-26.5), 'news_count': 30, 'sentiment_volatility': 0.7772, 'positive_ratio': 0.2333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.9674},
    {'company': 'LG유플러스', 'news_sentiment_score': np.float64(-26.74), 'news_count': 30, 'sentiment_volatility': 0.8259, 'positive_ratio': 0.3333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.9935},
    {'company': 'LG전자', 'news_sentiment_score': np.float64(-30.9), 'news_count': 30, 'sentiment_volatility': 0.3924, 'positive_ratio': 0.0, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9932},
    {'company': 'LG화학', 'news_sentiment_score': np.float64(-39.34), 'news_count': 30, 'sentiment_volatility': 0.5971, 'positive_ratio': 0.1, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9925},
    {'company': 'NAVER', 'news_sentiment_score': np.float64(-43.47), 'news_count': 30, 'sentiment_volatility': 0.6986, 'positive_ratio': 0.1667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9151},
    {'company': 'POSCO홀딩스', 'news_sentiment_score': np.float64(-20.59), 'news_count': 29, 'sentiment_volatility': 0.5827, 'positive_ratio': 0.1034, 'negative_ratio': 0.3448, 'recency_weight_mean': 0.8576},
    {'company': 'S-Oil', 'news_sentiment_score': np.float64(-32.61), 'news_count': 30, 'sentiment_volatility': 0.4244, 'positive_ratio': 0.0, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9667},
    {'company': 'SK이노베이션', 'news_sentiment_score': np.float64(-76.41), 'news_count': 30, 'sentiment_volatility': 0.5542, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9654},
    {'company': 'SK텔레콤', 'news_sentiment_score': np.float64(-21.99), 'news_count': 30, 'sentiment_volatility': 0.6838, 'positive_ratio': 0.1667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.9502},
    {'company': 'SK하이닉스', 'news_sentiment_score': np.float64(-38.77), 'news_count': 30, 'sentiment_volatility': 0.5166, 'positive_ratio': 0.0333, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.9896},
    {'company': '기아', 'news_sentiment_score': np.float64(-82.37), 'news_count': 30, 'sentiment_volatility': 0.3589, 'positive_ratio': 0.0, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9947},
    {'company': '대한항공', 'news_sentiment_score': np.float64(-65.37), 'news_count': 30, 'sentiment_volatility': 0.4652, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9954},
    {'company': '두산에너빌리티', 'news_sentiment_score': np.float64(8.06), 'news_count': 30, 'sentiment_volatility': 0.8043, 'positive_ratio': 0.3667, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9354},
    {'company': '롯데케미칼', 'news_sentiment_score': np.float64(-73.51), 'news_count': 30, 'sentiment_volatility': 0.4727, 'positive_ratio': 0.0333, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9195},
    {'company': '리더스코스메틱', 'news_sentiment_score': np.float64(-63.0), 'news_count': 30, 'sentiment_volatility': 0.4472, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.3426},
    {'company': '매일홀딩스', 'news_sentiment_score': np.float64(-31.87), 'news_count': 30, 'sentiment_volatility': 0.6265, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3417},
    {'company': '매커스', 'news_sentiment_score': np.float64(-68.13), 'news_count': 30, 'sentiment_volatility': 0.5479, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.4134},
    {'company': '멀티캠퍼스', 'news_sentiment_score': np.float64(-39.38), 'news_count': 30, 'sentiment_volatility': 0.5927, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.7042},
    {'company': '메디톡스', 'news_sentiment_score': np.float64(16.62), 'news_count': 30, 'sentiment_volatility': 0.4741, 'positive_ratio': 0.2333, 'negative_ratio': 0.0333, 'recency_weight_mean': 0.8375},
    {'company': '멕아이씨에스', 'news_sentiment_score': np.float64(-12.84), 'news_count': 29, 'sentiment_volatility': 0.4787, 'positive_ratio': 0.069, 'negative_ratio': 0.1724, 'recency_weight_mean': 0.2002},
    {'company': '명문제약', 'news_sentiment_score': np.float64(-36.67), 'news_count': 30, 'sentiment_volatility': 0.5961, 'positive_ratio': 0.1, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.611},
    {'company': '모나리자', 'news_sentiment_score': np.float64(-30.58), 'news_count': 30, 'sentiment_volatility': 0.8055, 'positive_ratio': 0.2333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.8877},
    {'company': '무학', 'news_sentiment_score': np.float64(-59.4), 'news_count': 30, 'sentiment_volatility': 0.5867, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9144},
    {'company': '문배철강', 'news_sentiment_score': np.float64(-46.61), 'news_count': 30, 'sentiment_volatility': 0.5422, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.4694},
    {'company': '미래산업', 'news_sentiment_score': np.float64(-51.25), 'news_count': 30, 'sentiment_volatility': 0.4648, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9889},
    {'company': '바이오니아', 'news_sentiment_score': np.float64(-59.34), 'news_count': 30, 'sentiment_volatility': 0.5463, 'positive_ratio': 0.0667, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.524},
    {'company': '바이오스마트', 'news_sentiment_score': np.float64(-39.66), 'news_count': 30, 'sentiment_volatility': 0.449, 'positive_ratio': 0.0333, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.5987},
    {'company': '방림', 'news_sentiment_score': np.float64(-88.29), 'news_count': 30, 'sentiment_volatility': 0.2797, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.7998},
    {'company': '베뉴지', 'news_sentiment_score': np.float64(-87.71), 'news_count': 30, 'sentiment_volatility': 0.7878, 'positive_ratio': 0.2667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.0839},
    {'company': '벽산', 'news_sentiment_score': np.float64(-63.64), 'news_count': 30, 'sentiment_volatility': 0.5794, 'positive_ratio': 0.0667, 'negative_ratio': 0.7, 'recency_weight_mean': 0.8834},
    {'company': '부산주공', 'news_sentiment_score': np.float64(-64.9), 'news_count': 30, 'sentiment_volatility': 0.5722, 'positive_ratio': 0.0667, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.4107},
    {'company': '부스타', 'news_sentiment_score': np.float64(-53.83), 'news_count': 30, 'sentiment_volatility': 0.5948, 'positive_ratio': 0.0667, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.2926},
    {'company': '브이티', 'news_sentiment_score': np.float64(-72.83), 'news_count': 29, 'sentiment_volatility': 0.5515, 'positive_ratio': 0.069, 'negative_ratio': 0.7241, 'recency_weight_mean': 0.5711},
    {'company': '비나텍', 'news_sentiment_score': np.float64(-43.64), 'news_count': 30, 'sentiment_volatility': 0.5228, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5161},
    {'company': '비에이치', 'news_sentiment_score': np.float64(-6.77), 'news_count': 30, 'sentiment_volatility': 0.4389, 'positive_ratio': 0.0667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.8261},
    {'company': '비엠티', 'news_sentiment_score': np.float64(-25.13), 'news_count': 30, 'sentiment_volatility': 0.5252, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5327},
    {'company': '사조대림', 'news_sentiment_score': np.float64(-50.07), 'news_count': 30, 'sentiment_volatility': 0.6014, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.7619},
    {'company': '사조산업', 'news_sentiment_score': np.float64(-37.65), 'news_count': 29, 'sentiment_volatility': 0.7747, 'positive_ratio': 0.2414, 'negative_ratio': 0.5172, 'recency_weight_mean': 0.6122},
    {'company': '사조오양', 'news_sentiment_score': np.float64(-61.68), 'news_count': 30, 'sentiment_volatility': 0.7854, 'positive_ratio': 0.2, 'negative_ratio': 0.6, 'recency_weight_mean': 0.2214},
    {'company': '삼목에스폼', 'news_sentiment_score': np.float64(-53.87), 'news_count': 30, 'sentiment_volatility': 0.5173, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.1361},
    {'company': '삼성E&A', 'news_sentiment_score': np.float64(-14.35), 'news_count': 30, 'sentiment_volatility': 0.4807, 'positive_ratio': 0.1, 'negative_ratio': 0.2, 'recency_weight_mean': 0.934},
    {'company': '삼성SDI', 'news_sentiment_score': np.float64(-56.26), 'news_count': 30, 'sentiment_volatility': 0.5271, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.9251},
    {'company': '삼성물산', 'news_sentiment_score': np.float64(-60.24), 'news_count': 30, 'sentiment_volatility': 0.5004, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.9775},
    {'company': '삼성바이오로직스', 'news_sentiment_score': np.float64(-54.7), 'news_count': 30, 'sentiment_volatility': 0.5037, 'positive_ratio': 0.0333, 'negative_ratio': 0.6, 'recency_weight_mean': 0.952},
    {'company': '삼성에스디에스', 'news_sentiment_score': np.float64(-36.67), 'news_count': 30, 'sentiment_volatility': 0.6384, 'positive_ratio': 0.1, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.8437},
    {'company': '삼성전자', 'news_sentiment_score': np.float64(-72.98), 'news_count': 30, 'sentiment_volatility': 0.4789, 'positive_ratio': 0.0333, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9928},
    {'company': '삼진엘앤디', 'news_sentiment_score': np.float64(-43.28), 'news_count': 30, 'sentiment_volatility': 0.6264, 'positive_ratio': 0.1, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.72},
    {'company': '삼천리자전거', 'news_sentiment_score': np.float64(-69.81), 'news_count': 30, 'sentiment_volatility': 0.4582, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.394},
    {'company': '삼현철강', 'news_sentiment_score': np.float64(-45.04), 'news_count': 30, 'sentiment_volatility': 0.5877, 'positive_ratio': 0.0667, 'negative_ratio': 0.5, 'recency_weight_mean': 0.5618},
    {'company': '삼화네트웍스', 'news_sentiment_score': np.float64(-68.44), 'news_count': 30, 'sentiment_volatility': 0.4436, 'positive_ratio': 0.0333, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9486},
    {'company': '상신브레이크', 'news_sentiment_score': np.float64(-66.48), 'news_count': 30, 'sentiment_volatility': 0.528, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.5166},
    {'company': '상신이디피', 'news_sentiment_score': np.float64(-47.0), 'news_count': 30, 'sentiment_volatility': 0.5887, 'positive_ratio': 0.1, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.529},
    {'company': '상지건설', 'news_sentiment_score': np.float64(-49.17), 'news_count': 30, 'sentiment_volatility': 0.6293, 'positive_ratio': 0.1333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4072},
    {'company': '샘표', 'news_sentiment_score': np.float64(-87.06), 'news_count': 30, 'sentiment_volatility': 0.3389, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9281},
    {'company': '서부T&D', 'news_sentiment_score': np.float64(-69.62), 'news_count': 30, 'sentiment_volatility': 0.4904, 'positive_ratio': 0.0, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4536},
    {'company': '서연', 'news_sentiment_score': np.float64(-88.77), 'news_count': 30, 'sentiment_volatility': 0.3007, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.7864},
    {'company': '서울반도체', 'news_sentiment_score': np.float64(-18.22), 'news_count': 30, 'sentiment_volatility': 0.7354, 'positive_ratio': 0.2667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.393},
    {'company': '서울식품', 'news_sentiment_score': np.float64(-47.24), 'news_count': 30, 'sentiment_volatility': 0.6167, 'positive_ratio': 0.1, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.8357},
    {'company': '서울제약', 'news_sentiment_score': np.float64(-44.15), 'news_count': 30, 'sentiment_volatility': 0.6813, 'positive_ratio': 0.1333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3608},
    {'company': '서한', 'news_sentiment_score': np.float64(-86.07), 'news_count': 30, 'sentiment_volatility': 0.2892, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9573},
    {'company': '선바이오', 'news_sentiment_score': np.float64(-15.98), 'news_count': 30, 'sentiment_volatility': 0.5882, 'positive_ratio': 0.1, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.0597},
    {'company': '선진뷰티사이언스', 'news_sentiment_score': np.float64(-50.14), 'news_count': 28, 'sentiment_volatility': 0.7392, 'positive_ratio': 0.2143, 'negative_ratio': 0.4643, 'recency_weight_mean': 0.2382},
    {'company': '성문전자', 'news_sentiment_score': np.float64(-33.28), 'news_count': 30, 'sentiment_volatility': 0.5536, 'positive_ratio': 0.0667, 'negative_ratio': 0.5, 'recency_weight_mean': 0.5842},
    {'company': '성안머티리얼스', 'news_sentiment_score': np.float64(-32.34), 'news_count': 30, 'sentiment_volatility': 0.5914, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4701},
    {'company': '성우전자', 'news_sentiment_score': np.float64(-46.5), 'news_count': 30, 'sentiment_volatility': 0.4486, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.4294},
    {'company': '세명전기', 'news_sentiment_score': np.float64(-20.9), 'news_count': 30, 'sentiment_volatility': 0.4003, 'positive_ratio': 0.0333, 'negative_ratio': 0.2, 'recency_weight_mean': 0.8269},
    {'company': '세방', 'news_sentiment_score': np.float64(-62.48), 'news_count': 30, 'sentiment_volatility': 0.713, 'positive_ratio': 0.1667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.7258},
    {'company': '세방전지', 'news_sentiment_score': np.float64(-24.94), 'news_count': 30, 'sentiment_volatility': 0.7881, 'positive_ratio': 0.3333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5506},
    {'company': '세아베스틸지주', 'news_sentiment_score': np.float64(14.59), 'news_count': 30, 'sentiment_volatility': 0.6153, 'positive_ratio': 0.2667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.5946},
    {'company': '셀트리온', 'news_sentiment_score': np.float64(-31.45), 'news_count': 30, 'sentiment_volatility': 0.5709, 'positive_ratio': 0.0667, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.9607},
    {'company': '소노스퀘어', 'news_sentiment_score': np.float64(-6.4), 'news_count': 29, 'sentiment_volatility': 0.4603, 'positive_ratio': 0.0345, 'negative_ratio': 0.2414, 'recency_weight_mean': 0.178},
    {'company': '쇼박스', 'news_sentiment_score': np.float64(-62.53), 'news_count': 30, 'sentiment_volatility': 0.5812, 'positive_ratio': 0.0667, 'negative_ratio': 0.7, 'recency_weight_mean': 0.9151},
    {'company': '수산세보틱스', 'news_sentiment_score': np.float64(-76.51), 'news_count': 30, 'sentiment_volatility': 0.4715, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.111},
    {'company': '슈프리마에이치큐', 'news_sentiment_score': np.float64(-29.87), 'news_count': 30, 'sentiment_volatility': 0.5555, 'positive_ratio': 0.0667, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.3846},
    {'company': '신라교역', 'news_sentiment_score': np.float64(-14.45), 'news_count': 30, 'sentiment_volatility': 0.6886, 'positive_ratio': 0.1333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3568},
    {'company': '신성이엔지', 'news_sentiment_score': np.float64(-38.36), 'news_count': 30, 'sentiment_volatility': 0.5728, 'positive_ratio': 0.0667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.8654},
    {'company': '신진에스엠', 'news_sentiment_score': np.float64(-18.35), 'news_count': 30, 'sentiment_volatility': 0.5794, 'positive_ratio': 0.1333, 'negative_ratio': 0.3, 'recency_weight_mean': 0.3402},
    {'company': '싸이토젠', 'news_sentiment_score': np.float64(-48.8), 'news_count': 30, 'sentiment_volatility': 0.5235, 'positive_ratio': 0.0333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.2693},
    {'company': '쎌바이오텍', 'news_sentiment_score': np.float64(-57.52), 'news_count': 30, 'sentiment_volatility': 0.5224, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.1764},
    {'company': '씨큐브', 'news_sentiment_score': np.float64(-20.59), 'news_count': 28, 'sentiment_volatility': 0.5447, 'positive_ratio': 0.0357, 'negative_ratio': 0.6071, 'recency_weight_mean': 0.2022},
    {'company': '아모레퍼시픽', 'news_sentiment_score': np.float64(-70.01), 'news_count': 30, 'sentiment_volatility': 0.408, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.9453},
    {'company': '아시아나항공', 'news_sentiment_score': np.float64(-77.65), 'news_count': 30, 'sentiment_volatility': 0.5386, 'positive_ratio': 0.0667, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9127},
    {'company': '아이씨티케이', 'news_sentiment_score': np.float64(-62.49), 'news_count': 30, 'sentiment_volatility': 0.4783, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.5416},
    {'company': '아이엠', 'news_sentiment_score': np.float64(-64.04), 'news_count': 30, 'sentiment_volatility': 0.6412, 'positive_ratio': 0.1, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.8743},
    {'company': '아이즈비전', 'news_sentiment_score': np.float64(-59.66), 'news_count': 30, 'sentiment_volatility': 0.4472, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3579},
    {'company': '아이티센엔텍', 'news_sentiment_score': np.float64(-46.94), 'news_count': 29, 'sentiment_volatility': 0.5188, 'positive_ratio': 0.069, 'negative_ratio': 0.5172, 'recency_weight_mean': 0.7973},
    {'company': '아이패밀리에스씨', 'news_sentiment_score': np.float64(-8.35), 'news_count': 30, 'sentiment_volatility': 0.6402, 'positive_ratio': 0.1667, 'negative_ratio': 0.3, 'recency_weight_mean': 0.7621},
    {'company': '아주스틸', 'news_sentiment_score': np.float64(-30.64), 'news_count': 30, 'sentiment_volatility': 0.7815, 'positive_ratio': 0.2667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.6537},
    {'company': '알에프텍', 'news_sentiment_score': np.float64(-62.23), 'news_count': 30, 'sentiment_volatility': 0.6686, 'positive_ratio': 0.1667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.4489},
    {'company': '알티캐스트', 'news_sentiment_score': np.float64(-39.21), 'news_count': 30, 'sentiment_volatility': 0.5847, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.1632},
    {'company': '에스디생명공학', 'news_sentiment_score': np.float64(-45.32), 'news_count': 30, 'sentiment_volatility': 0.8164, 'positive_ratio': 0.2667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3111},
    {'company': '에스디시스템', 'news_sentiment_score': np.float64(-51.33), 'news_count': 30, 'sentiment_volatility': 0.4414, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.0341},
    {'company': '에스앤더블류', 'news_sentiment_score': np.float64(-31.47), 'news_count': 30, 'sentiment_volatility': 0.5007, 'positive_ratio': 0.0333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.4891},
    {'company': '에스에이엠티', 'news_sentiment_score': np.float64(-42.36), 'news_count': 30, 'sentiment_volatility': 0.4376, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.4054},
    {'company': '에스에이티', 'news_sentiment_score': np.float64(-65.47), 'news_count': 29, 'sentiment_volatility': 0.5323, 'positive_ratio': 0.0345, 'negative_ratio': 0.4828, 'recency_weight_mean': 0.0929},
    {'company': '에스피지', 'news_sentiment_score': np.float64(-54.42), 'news_count': 30, 'sentiment_volatility': 0.5412, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.5207},
    {'company': '에쓰씨엔지니어링', 'news_sentiment_score': np.float64(-75.62), 'news_count': 29, 'sentiment_volatility': 0.4411, 'positive_ratio': 0.0, 'negative_ratio': 0.6552, 'recency_weight_mean': 0.1949},
    {'company': '에이디테크놀로지', 'news_sentiment_score': np.float64(-47.25), 'news_count': 30, 'sentiment_volatility': 0.5413, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.881},
    {'company': '에코프로', 'news_sentiment_score': np.float64(6.19), 'news_count': 30, 'sentiment_volatility': 0.6504, 'positive_ratio': 0.2667, 'negative_ratio': 0.2, 'recency_weight_mean': 0.9808},
    {'company': '에프에스티', 'news_sentiment_score': np.float64(-45.28), 'news_count': 30, 'sentiment_volatility': 0.4786, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.3228},
    {'company': '엔씨소프트', 'news_sentiment_score': np.float64(-80.01), 'news_count': 30, 'sentiment_volatility': 0.4343, 'positive_ratio': 0.0333, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9842},
    {'company': '엔에스이엔엠', 'news_sentiment_score': np.float64(-45.01), 'news_count': 30, 'sentiment_volatility': 0.5351, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.4751},
    {'company': '엠에스씨', 'news_sentiment_score': np.float64(-63.28), 'news_count': 30, 'sentiment_volatility': 0.4751, 'positive_ratio': 0.0, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3145},
    {'company': '엠케이전자', 'news_sentiment_score': np.float64(-10.06), 'news_count': 30, 'sentiment_volatility': 0.4979, 'positive_ratio': 0.1, 'negative_ratio': 0.1667, 'recency_weight_mean': 0.7487},
    {'company': '영원무역', 'news_sentiment_score': np.float64(-14.52), 'news_count': 30, 'sentiment_volatility': 0.4541, 'positive_ratio': 0.0667, 'negative_ratio': 0.2, 'recency_weight_mean': 0.7458},
    {'company': '영풍', 'news_sentiment_score': np.float64(22.92), 'news_count': 30, 'sentiment_volatility': 0.8054, 'positive_ratio': 0.5667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.9878},
    {'company': '예스티', 'news_sentiment_score': np.float64(-16.99), 'news_count': 30, 'sentiment_volatility': 0.5472, 'positive_ratio': 0.0667, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3871},
    {'company': '오공', 'news_sentiment_score': np.float64(-53.68), 'news_count': 30, 'sentiment_volatility': 0.5134, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.571},
    {'company': '오뚜기', 'news_sentiment_score': np.float64(-99.8), 'news_count': 30, 'sentiment_volatility': 0.0052, 'positive_ratio': 0.0, 'negative_ratio': 1.0, 'recency_weight_mean': 0.9865},
    {'company': '오로라', 'news_sentiment_score': np.float64(-90.47), 'news_count': 30, 'sentiment_volatility': 0.2622, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.9315},
    {'company': '오리콤', 'news_sentiment_score': np.float64(-58.07), 'news_count': 30, 'sentiment_volatility': 0.5689, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.458},
    {'company': '오텍', 'news_sentiment_score': np.float64(-31.61), 'news_count': 30, 'sentiment_volatility': 0.4774, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5738},
    {'company': '옴니시스템', 'news_sentiment_score': np.float64(-57.36), 'news_count': 30, 'sentiment_volatility': 0.5906, 'positive_ratio': 0.1, 'negative_ratio': 0.3, 'recency_weight_mean': 0.1729},
    {'company': '옵트론텍', 'news_sentiment_score': np.float64(-42.05), 'news_count': 30, 'sentiment_volatility': 0.4538, 'positive_ratio': 0.0, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.8716},
    {'company': '와이어블', 'news_sentiment_score': np.float64(-42.82), 'news_count': 30, 'sentiment_volatility': 0.4698, 'positive_ratio': 0.0, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3306},
    {'company': '와이즈버즈', 'news_sentiment_score': np.float64(-59.91), 'news_count': 30, 'sentiment_volatility': 0.4413, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.8564},
    {'company': '와이지-원', 'news_sentiment_score': np.float64(-72.12), 'news_count': 30, 'sentiment_volatility': 0.5245, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.6341},
    {'company': '우리기술투자', 'news_sentiment_score': np.float64(-70.28), 'news_count': 30, 'sentiment_volatility': 0.5661, 'positive_ratio': 0.0667, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.9126},
    {'company': '우성', 'news_sentiment_score': np.float64(-78.85), 'news_count': 30, 'sentiment_volatility': 0.3568, 'positive_ratio': 0.0, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9651},
    {'company': '우신시스템', 'news_sentiment_score': np.float64(-64.77), 'news_count': 30, 'sentiment_volatility': 0.4452, 'positive_ratio': 0.0, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.6908},
    {'company': '우양에이치씨', 'news_sentiment_score': np.float64(-84.78), 'news_count': 30, 'sentiment_volatility': 0.606, 'positive_ratio': 0.1, 'negative_ratio': 0.4, 'recency_weight_mean': 0.2102},
    {'company': '우주일렉트로', 'news_sentiment_score': np.float64(-76.31), 'news_count': 30, 'sentiment_volatility': 0.5447, 'positive_ratio': 0.0333, 'negative_ratio': 0.6, 'recency_weight_mean': 0.342},
    {'company': '원림', 'news_sentiment_score': np.float64(-80.87), 'news_count': 27, 'sentiment_volatility': 0.4762, 'positive_ratio': 0.0, 'negative_ratio': 0.6296, 'recency_weight_mean': 0.498},
    {'company': '원익큐브', 'news_sentiment_score': np.float64(-39.68), 'news_count': 30, 'sentiment_volatility': 0.4788, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.3414},
    {'company': '월덱스', 'news_sentiment_score': np.float64(-34.23), 'news_count': 30, 'sentiment_volatility': 0.5289, 'positive_ratio': 0.0333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.2946},
    {'company': '웰크론', 'news_sentiment_score': np.float64(-53.41), 'news_count': 30, 'sentiment_volatility': 0.4546, 'positive_ratio': 0.0, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.4351},
    {'company': '유니온머티리얼', 'news_sentiment_score': np.float64(-1.85), 'news_count': 30, 'sentiment_volatility': 0.3818, 'positive_ratio': 0.0667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.6583},
    {'company': '유니켐', 'news_sentiment_score': np.float64(-52.78), 'news_count': 29, 'sentiment_volatility': 0.4614, 'positive_ratio': 0.0, 'negative_ratio': 0.4828, 'recency_weight_mean': 0.1427},
    {'company': '유니테스트', 'news_sentiment_score': np.float64(-43.8), 'news_count': 30, 'sentiment_volatility': 0.6304, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.7025},
    {'company': '유비온', 'news_sentiment_score': np.float64(-79.69), 'news_count': 30, 'sentiment_volatility': 0.4026, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.2372},
    {'company': '유비쿼스홀딩스', 'news_sentiment_score': np.float64(-52.88), 'news_count': 30, 'sentiment_volatility': 0.5552, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3701},
    {'company': '인스코비', 'news_sentiment_score': np.float64(-41.42), 'news_count': 30, 'sentiment_volatility': 0.5239, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.6891},
    {'company': '인지소프트', 'news_sentiment_score': np.float64(-24.76), 'news_count': 30, 'sentiment_volatility': 0.3985, 'positive_ratio': 0.0, 'negative_ratio': 0.2, 'recency_weight_mean': 0.1151},
    {'company': '인지컨트롤스', 'news_sentiment_score': np.float64(-37.67), 'news_count': 30, 'sentiment_volatility': 0.539, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.5996},
    {'company': '인텍플러스', 'news_sentiment_score': np.float64(-39.83), 'news_count': 30, 'sentiment_volatility': 0.566, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.9932}
]

# 2. DataFrame으로 변환
df = pd.DataFrame(records)

# 3. CSV 파일로 저장
# encoding="utf-8-sig"는 Excel에서 한글이 깨지지 않게 보장해 줍니다.
df.to_csv("news_features.csv", index=False, encoding="utf-8-sig")

print("\n✅ 모든 기업 분석 완료 → news_features.csv 저장됨")

# (선택 사항) 저장된 파일 내용 확인
print("\n--- [news_features.csv 파일 내용 미리보기] ---")
print(df.head())

NameError: name 'np' is not defined

In [2]:
import pandas as pd
import numpy as np
import io

# Provided data from the analysis results
data = [
    {'company': 'CJ', 'news_sentiment_score': np.float64(-52.92), 'news_count': 30, 'sentiment_volatility': 0.4649, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9921},
    {'company': 'CJ ENM', 'news_sentiment_score': np.float64(-42.92), 'news_count': 30, 'sentiment_volatility': 0.5189, 'positive_ratio': 0.0333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.8867},
    {'company': 'CJ제일제당', 'news_sentiment_score': np.float64(-13.13), 'news_count': 30, 'sentiment_volatility': 0.5684, 'positive_ratio': 0.1333, 'negative_ratio': 0.2667, 'recency_weight_mean': 0.9185},
    {'company': 'HMM', 'news_sentiment_score': np.float64(-76.41), 'news_count': 30, 'sentiment_volatility': 0.5532, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9906},
    {'company': 'KT', 'news_sentiment_score': np.float64(-38.93), 'news_count': 29, 'sentiment_volatility': 0.826, 'positive_ratio': 0.2414, 'negative_ratio': 0.6207, 'recency_weight_mean': 0.987},
    {'company': 'KT&G', 'news_sentiment_score': np.float64(-46.87), 'news_count': 30, 'sentiment_volatility': 0.552, 'positive_ratio': 0.0333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.9173},
    {'company': 'LG생활건강', 'news_sentiment_score': np.float64(-74.43), 'news_count': 30, 'sentiment_volatility': 0.5427, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9612},
    {'company': 'LG에너지솔루션', 'news_sentiment_score': np.float64(-26.5), 'news_count': 30, 'sentiment_volatility': 0.7772, 'positive_ratio': 0.2333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.9674},
    {'company': 'LG유플러스', 'news_sentiment_score': np.float64(-26.74), 'news_count': 30, 'sentiment_volatility': 0.8259, 'positive_ratio': 0.3333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.9935},
    {'company': 'LG전자', 'news_sentiment_score': np.float64(-30.9), 'news_count': 30, 'sentiment_volatility': 0.3924, 'positive_ratio': 0.0, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9932},
    {'company': 'LG화학', 'news_sentiment_score': np.float64(-39.34), 'news_count': 30, 'sentiment_volatility': 0.5971, 'positive_ratio': 0.1, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9925},
    {'company': 'NAVER', 'news_sentiment_score': np.float64(-43.47), 'news_count': 30, 'sentiment_volatility': 0.6986, 'positive_ratio': 0.1667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9151},
    {'company': 'POSCO홀딩스', 'news_sentiment_score': np.float64(-20.59), 'news_count': 29, 'sentiment_volatility': 0.5827, 'positive_ratio': 0.1034, 'negative_ratio': 0.3448, 'recency_weight_mean': 0.8576},
    {'company': 'S-Oil', 'news_sentiment_score': np.float64(-32.61), 'news_count': 30, 'sentiment_volatility': 0.4244, 'positive_ratio': 0.0, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9667},
    {'company': 'SK이노베이션', 'news_sentiment_score': np.float64(-76.41), 'news_count': 30, 'sentiment_volatility': 0.5542, 'positive_ratio': 0.0667, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9654},
    {'company': 'SK텔레콤', 'news_sentiment_score': np.float64(-21.99), 'news_count': 30, 'sentiment_volatility': 0.6838, 'positive_ratio': 0.1667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.9502},
    {'company': 'SK하이닉스', 'news_sentiment_score': np.float64(-38.77), 'news_count': 30, 'sentiment_volatility': 0.5166, 'positive_ratio': 0.0333, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.9896},
    {'company': '기아', 'news_sentiment_score': np.float64(-82.37), 'news_count': 30, 'sentiment_volatility': 0.3589, 'positive_ratio': 0.0, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9947},
    {'company': '대한항공', 'news_sentiment_score': np.float64(-65.37), 'news_count': 30, 'sentiment_volatility': 0.4652, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9954},
    {'company': '두산에너빌리티', 'news_sentiment_score': np.float64(8.06), 'news_count': 30, 'sentiment_volatility': 0.8043, 'positive_ratio': 0.3667, 'negative_ratio': 0.3, 'recency_weight_mean': 0.9354},
    {'company': '롯데케미칼', 'news_sentiment_score': np.float64(-73.51), 'news_count': 30, 'sentiment_volatility': 0.4727, 'positive_ratio': 0.0333, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9195},
    {'company': '리더스코스메틱', 'news_sentiment_score': np.float64(-63.0), 'news_count': 30, 'sentiment_volatility': 0.4472, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.3426},
    {'company': '매일홀딩스', 'news_sentiment_score': np.float64(-31.87), 'news_count': 30, 'sentiment_volatility': 0.6265, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3417},
    {'company': '매커스', 'news_sentiment_score': np.float64(-68.13), 'news_count': 30, 'sentiment_volatility': 0.5479, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.4134},
    {'company': '멀티캠퍼스', 'news_sentiment_score': np.float64(-39.38), 'news_count': 30, 'sentiment_volatility': 0.5927, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.7042},
    {'company': '메디톡스', 'news_sentiment_score': np.float64(16.62), 'news_count': 30, 'sentiment_volatility': 0.4741, 'positive_ratio': 0.2333, 'negative_ratio': 0.0333, 'recency_weight_mean': 0.8375},
    {'company': '멕아이씨에스', 'news_sentiment_score': np.float64(-12.84), 'news_count': 29, 'sentiment_volatility': 0.4787, 'positive_ratio': 0.069, 'negative_ratio': 0.1724, 'recency_weight_mean': 0.2002},
    {'company': '명문제약', 'news_sentiment_score': np.float64(-36.67), 'news_count': 30, 'sentiment_volatility': 0.5961, 'positive_ratio': 0.1, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.611},
    {'company': '모나리자', 'news_sentiment_score': np.float64(-30.58), 'news_count': 30, 'sentiment_volatility': 0.8055, 'positive_ratio': 0.2333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.8877},
    {'company': '무학', 'news_sentiment_score': np.float64(-59.4), 'news_count': 30, 'sentiment_volatility': 0.5867, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9144},
    {'company': '문배철강', 'news_sentiment_score': np.float64(-46.61), 'news_count': 30, 'sentiment_volatility': 0.5422, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.4694},
    {'company': '미래산업', 'news_sentiment_score': np.float64(-51.25), 'news_count': 30, 'sentiment_volatility': 0.4648, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9889},
    {'company': '바이오니아', 'news_sentiment_score': np.float64(-59.34), 'news_count': 30, 'sentiment_volatility': 0.5463, 'positive_ratio': 0.0667, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.524},
    {'company': '바이오스마트', 'news_sentiment_score': np.float64(-39.66), 'news_count': 30, 'sentiment_volatility': 0.449, 'positive_ratio': 0.0333, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.5987},
    {'company': '방림', 'news_sentiment_score': np.float64(-88.29), 'news_count': 30, 'sentiment_volatility': 0.2797, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.7998},
    {'company': '베뉴지', 'news_sentiment_score': np.float64(-87.71), 'news_count': 30, 'sentiment_volatility': 0.7878, 'positive_ratio': 0.2667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.0839},
    {'company': '벽산', 'news_sentiment_score': np.float64(-63.64), 'news_count': 30, 'sentiment_volatility': 0.5794, 'positive_ratio': 0.0667, 'negative_ratio': 0.7, 'recency_weight_mean': 0.8834},
    {'company': '부산주공', 'news_sentiment_score': np.float64(-64.9), 'news_count': 30, 'sentiment_volatility': 0.5722, 'positive_ratio': 0.0667, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.4107},
    {'company': '부스타', 'news_sentiment_score': np.float64(-53.83), 'news_count': 30, 'sentiment_volatility': 0.5948, 'positive_ratio': 0.0667, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.2926},
    {'company': '브이티', 'news_sentiment_score': np.float64(-72.83), 'news_count': 29, 'sentiment_volatility': 0.5515, 'positive_ratio': 0.069, 'negative_ratio': 0.7241, 'recency_weight_mean': 0.5711},
    {'company': '비나텍', 'news_sentiment_score': np.float64(-43.64), 'news_count': 30, 'sentiment_volatility': 0.5228, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5161},
    {'company': '비에이치', 'news_sentiment_score': np.float64(-6.77), 'news_count': 30, 'sentiment_volatility': 0.4389, 'positive_ratio': 0.0667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.8261},
    {'company': '비엠티', 'news_sentiment_score': np.float64(-25.13), 'news_count': 30, 'sentiment_volatility': 0.5252, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5327},
    {'company': '사조대림', 'news_sentiment_score': np.float64(-50.07), 'news_count': 30, 'sentiment_volatility': 0.6014, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.7619},
    {'company': '사조산업', 'news_sentiment_score': np.float64(-37.65), 'news_count': 29, 'sentiment_volatility': 0.7747, 'positive_ratio': 0.2414, 'negative_ratio': 0.5172, 'recency_weight_mean': 0.6122},
    {'company': '사조오양', 'news_sentiment_score': np.float64(-61.68), 'news_count': 30, 'sentiment_volatility': 0.7854, 'positive_ratio': 0.2, 'negative_ratio': 0.6, 'recency_weight_mean': 0.2214},
    {'company': '삼목에스폼', 'news_sentiment_score': np.float64(-53.87), 'news_count': 30, 'sentiment_volatility': 0.5173, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.1361},
    {'company': '삼성E&A', 'news_sentiment_score': np.float64(-14.35), 'news_count': 30, 'sentiment_volatility': 0.4807, 'positive_ratio': 0.1, 'negative_ratio': 0.2, 'recency_weight_mean': 0.934},
    {'company': '삼성SDI', 'news_sentiment_score': np.float64(-56.26), 'news_count': 30, 'sentiment_volatility': 0.5271, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.9251},
    {'company': '삼성물산', 'news_sentiment_score': np.float64(-60.24), 'news_count': 30, 'sentiment_volatility': 0.5004, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.9775},
    {'company': '삼성바이오로직스', 'news_sentiment_score': np.float64(-54.7), 'news_count': 30, 'sentiment_volatility': 0.5037, 'positive_ratio': 0.0333, 'negative_ratio': 0.6, 'recency_weight_mean': 0.952},
    {'company': '삼성에스디에스', 'news_sentiment_score': np.float64(-36.67), 'news_count': 30, 'sentiment_volatility': 0.6384, 'positive_ratio': 0.1, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.8437},
    {'company': '삼성전자', 'news_sentiment_score': np.float64(-72.98), 'news_count': 30, 'sentiment_volatility': 0.4789, 'positive_ratio': 0.0333, 'negative_ratio': 0.8333, 'recency_weight_mean': 0.9928},
    {'company': '삼진엘앤디', 'news_sentiment_score': np.float64(-43.28), 'news_count': 30, 'sentiment_volatility': 0.6264, 'positive_ratio': 0.1, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.72},
    {'company': '삼천리자전거', 'news_sentiment_score': np.float64(-69.81), 'news_count': 30, 'sentiment_volatility': 0.4582, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.394},
    {'company': '삼현철강', 'news_sentiment_score': np.float64(-45.04), 'news_count': 30, 'sentiment_volatility': 0.5877, 'positive_ratio': 0.0667, 'negative_ratio': 0.5, 'recency_weight_mean': 0.5618},
    {'company': '삼화네트웍스', 'news_sentiment_score': np.float64(-68.44), 'news_count': 30, 'sentiment_volatility': 0.4436, 'positive_ratio': 0.0333, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.9486},
    {'company': '상신브레이크', 'news_sentiment_score': np.float64(-66.48), 'news_count': 30, 'sentiment_volatility': 0.528, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.5166},
    {'company': '상신이디피', 'news_sentiment_score': np.float64(-47.0), 'news_count': 30, 'sentiment_volatility': 0.5887, 'positive_ratio': 0.1, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.529},
    {'company': '상지건설', 'news_sentiment_score': np.float64(-49.17), 'news_count': 30, 'sentiment_volatility': 0.6293, 'positive_ratio': 0.1333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4072},
    {'company': '샘표', 'news_sentiment_score': np.float64(-87.06), 'news_count': 30, 'sentiment_volatility': 0.3389, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9281},
    {'company': '서부T&D', 'news_sentiment_score': np.float64(-69.62), 'news_count': 30, 'sentiment_volatility': 0.4904, 'positive_ratio': 0.0, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4536},
    {'company': '서연', 'news_sentiment_score': np.float64(-88.77), 'news_count': 30, 'sentiment_volatility': 0.3007, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.7864},
    {'company': '서울반도체', 'news_sentiment_score': np.float64(-18.22), 'news_count': 30, 'sentiment_volatility': 0.7354, 'positive_ratio': 0.2667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.393},
    {'company': '서울식품', 'news_sentiment_score': np.float64(-47.24), 'news_count': 30, 'sentiment_volatility': 0.6167, 'positive_ratio': 0.1, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.8357},
    {'company': '서울제약', 'news_sentiment_score': np.float64(-44.15), 'news_count': 30, 'sentiment_volatility': 0.6813, 'positive_ratio': 0.1333, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3608},
    {'company': '서한', 'news_sentiment_score': np.float64(-86.07), 'news_count': 30, 'sentiment_volatility': 0.2892, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9573},
    {'company': '선바이오', 'news_sentiment_score': np.float64(-15.98), 'news_count': 30, 'sentiment_volatility': 0.5882, 'positive_ratio': 0.1, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.0597},
    {'company': '선진뷰티사이언스', 'news_sentiment_score': np.float64(-50.14), 'news_count': 28, 'sentiment_volatility': 0.7392, 'positive_ratio': 0.2143, 'negative_ratio': 0.4643, 'recency_weight_mean': 0.2382},
    {'company': '성문전자', 'news_sentiment_score': np.float64(-33.28), 'news_count': 30, 'sentiment_volatility': 0.5536, 'positive_ratio': 0.0667, 'negative_ratio': 0.5, 'recency_weight_mean': 0.5842},
    {'company': '성안머티리얼스', 'news_sentiment_score': np.float64(-32.34), 'news_count': 30, 'sentiment_volatility': 0.5914, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.4701},
    {'company': '성우전자', 'news_sentiment_score': np.float64(-46.5), 'news_count': 30, 'sentiment_volatility': 0.4486, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.4294},
    {'company': '세명전기', 'news_sentiment_score': np.float64(-20.9), 'news_count': 30, 'sentiment_volatility': 0.4003, 'positive_ratio': 0.0333, 'negative_ratio': 0.2, 'recency_weight_mean': 0.8269},
    {'company': '세방', 'news_sentiment_score': np.float64(-62.48), 'news_count': 30, 'sentiment_volatility': 0.713, 'positive_ratio': 0.1667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.7258},
    {'company': '세방전지', 'news_sentiment_score': np.float64(-24.94), 'news_count': 30, 'sentiment_volatility': 0.7881, 'positive_ratio': 0.3333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5506},
    {'company': '세아베스틸지주', 'news_sentiment_score': np.float64(14.59), 'news_count': 30, 'sentiment_volatility': 0.6153, 'positive_ratio': 0.2667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.5946},
    {'company': '셀트리온', 'news_sentiment_score': np.float64(-31.45), 'news_count': 30, 'sentiment_volatility': 0.5709, 'positive_ratio': 0.0667, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.9607},
    {'company': '소노스퀘어', 'news_sentiment_score': np.float64(-6.4), 'news_count': 29, 'sentiment_volatility': 0.4603, 'positive_ratio': 0.0345, 'negative_ratio': 0.2414, 'recency_weight_mean': 0.178},
    {'company': '쇼박스', 'news_sentiment_score': np.float64(-62.53), 'news_count': 30, 'sentiment_volatility': 0.5812, 'positive_ratio': 0.0667, 'negative_ratio': 0.7, 'recency_weight_mean': 0.9151},
    {'company': '수산세보틱스', 'news_sentiment_score': np.float64(-76.51), 'news_count': 30, 'sentiment_volatility': 0.4715, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.111},
    {'company': '슈프리마에이치큐', 'news_sentiment_score': np.float64(-29.87), 'news_count': 30, 'sentiment_volatility': 0.5555, 'positive_ratio': 0.0667, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.3846},
    {'company': '신라교역', 'news_sentiment_score': np.float64(-14.45), 'news_count': 30, 'sentiment_volatility': 0.6886, 'positive_ratio': 0.1333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3568},
    {'company': '신성이엔지', 'news_sentiment_score': np.float64(-38.36), 'news_count': 30, 'sentiment_volatility': 0.5728, 'positive_ratio': 0.0667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.8654},
    {'company': '신진에스엠', 'news_sentiment_score': np.float64(-18.35), 'news_count': 30, 'sentiment_volatility': 0.5794, 'positive_ratio': 0.1333, 'negative_ratio': 0.3, 'recency_weight_mean': 0.3402},
    {'company': '싸이토젠', 'news_sentiment_score': np.float64(-48.8), 'news_count': 30, 'sentiment_volatility': 0.5235, 'positive_ratio': 0.0333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.2693},
    {'company': '쎌바이오텍', 'news_sentiment_score': np.float64(-57.52), 'news_count': 30, 'sentiment_volatility': 0.5224, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.1764},
    {'company': '씨큐브', 'news_sentiment_score': np.float64(-20.59), 'news_count': 28, 'sentiment_volatility': 0.5447, 'positive_ratio': 0.0357, 'negative_ratio': 0.6071, 'recency_weight_mean': 0.2022},
    {'company': '아모레퍼시픽', 'news_sentiment_score': np.float64(-70.01), 'news_count': 30, 'sentiment_volatility': 0.408, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.9453},
    {'company': '아시아나항공', 'news_sentiment_score': np.float64(-77.65), 'news_count': 30, 'sentiment_volatility': 0.5386, 'positive_ratio': 0.0667, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9127},
    {'company': '아이씨티케이', 'news_sentiment_score': np.float64(-62.49), 'news_count': 30, 'sentiment_volatility': 0.4783, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.5416},
    {'company': '아이엠', 'news_sentiment_score': np.float64(-64.04), 'news_count': 30, 'sentiment_volatility': 0.6412, 'positive_ratio': 0.1, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.8743},
    {'company': '아이즈비전', 'news_sentiment_score': np.float64(-59.66), 'news_count': 30, 'sentiment_volatility': 0.4472, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3579},
    {'company': '아이티센엔텍', 'news_sentiment_score': np.float64(-46.94), 'news_count': 29, 'sentiment_volatility': 0.5188, 'positive_ratio': 0.069, 'negative_ratio': 0.5172, 'recency_weight_mean': 0.7973},
    {'company': '아이패밀리에스씨', 'news_sentiment_score': np.float64(-8.35), 'news_count': 30, 'sentiment_volatility': 0.6402, 'positive_ratio': 0.1667, 'negative_ratio': 0.3, 'recency_weight_mean': 0.7621},
    {'company': '아주스틸', 'news_sentiment_score': np.float64(-30.64), 'news_count': 30, 'sentiment_volatility': 0.7815, 'positive_ratio': 0.2667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.6537},
    {'company': '알에프텍', 'news_sentiment_score': np.float64(-62.23), 'news_count': 30, 'sentiment_volatility': 0.6686, 'positive_ratio': 0.1667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.4489},
    {'company': '알티캐스트', 'news_sentiment_score': np.float64(-39.21), 'news_count': 30, 'sentiment_volatility': 0.5847, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.1632},
    {'company': '에스디생명공학', 'news_sentiment_score': np.float64(-45.32), 'news_count': 30, 'sentiment_volatility': 0.8164, 'positive_ratio': 0.2667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3111},
    {'company': '에스디시스템', 'news_sentiment_score': np.float64(-51.33), 'news_count': 30, 'sentiment_volatility': 0.4414, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.0341},
    {'company': '에스앤더블류', 'news_sentiment_score': np.float64(-31.47), 'news_count': 30, 'sentiment_volatility': 0.5007, 'positive_ratio': 0.0333, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.4891},
    {'company': '에스에이엠티', 'news_sentiment_score': np.float64(-42.36), 'news_count': 30, 'sentiment_volatility': 0.4376, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.4054},
    {'company': '에스에이티', 'news_sentiment_score': np.float64(-65.47), 'news_count': 29, 'sentiment_volatility': 0.5323, 'positive_ratio': 0.0345, 'negative_ratio': 0.4828, 'recency_weight_mean': 0.0929},
    {'company': '에스피지', 'news_sentiment_score': np.float64(-54.42), 'news_count': 30, 'sentiment_volatility': 0.5412, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.5207},
    {'company': '에쓰씨엔지니어링', 'news_sentiment_score': np.float64(-75.62), 'news_count': 29, 'sentiment_volatility': 0.4411, 'positive_ratio': 0.0, 'negative_ratio': 0.6552, 'recency_weight_mean': 0.1949},
    {'company': '에이디테크놀로지', 'news_sentiment_score': np.float64(-47.25), 'news_count': 30, 'sentiment_volatility': 0.5413, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.881},
    {'company': '에코프로', 'news_sentiment_score': np.float64(6.19), 'news_count': 30, 'sentiment_volatility': 0.6504, 'positive_ratio': 0.2667, 'negative_ratio': 0.2, 'recency_weight_mean': 0.9808},
    {'company': '에프에스티', 'news_sentiment_score': np.float64(-45.28), 'news_count': 30, 'sentiment_volatility': 0.4786, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.3228},
    {'company': '엔씨소프트', 'news_sentiment_score': np.float64(-80.01), 'news_count': 30, 'sentiment_volatility': 0.4343, 'positive_ratio': 0.0333, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9842},
    {'company': '엔에스이엔엠', 'news_sentiment_score': np.float64(-45.01), 'news_count': 30, 'sentiment_volatility': 0.5351, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.4751},
    {'company': '엠에스씨', 'news_sentiment_score': np.float64(-63.28), 'news_count': 30, 'sentiment_volatility': 0.4751, 'positive_ratio': 0.0, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3145},
    {'company': '엠케이전자', 'news_sentiment_score': np.float64(-10.06), 'news_count': 30, 'sentiment_volatility': 0.4979, 'positive_ratio': 0.1, 'negative_ratio': 0.1667, 'recency_weight_mean': 0.7487},
    {'company': '영원무역', 'news_sentiment_score': np.float64(-14.52), 'news_count': 30, 'sentiment_volatility': 0.4541, 'positive_ratio': 0.0667, 'negative_ratio': 0.2, 'recency_weight_mean': 0.7458},
    {'company': '영풍', 'news_sentiment_score': np.float64(22.92), 'news_count': 30, 'sentiment_volatility': 0.8054, 'positive_ratio': 0.5667, 'negative_ratio': 0.4, 'recency_weight_mean': 0.9878},
    {'company': '예스티', 'news_sentiment_score': np.float64(-16.99), 'news_count': 30, 'sentiment_volatility': 0.5472, 'positive_ratio': 0.0667, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.3871},
    {'company': '오공', 'news_sentiment_score': np.float64(-53.68), 'news_count': 30, 'sentiment_volatility': 0.5134, 'positive_ratio': 0.0667, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.571},
    {'company': '오뚜기', 'news_sentiment_score': np.float64(-99.8), 'news_count': 30, 'sentiment_volatility': 0.0052, 'positive_ratio': 0.0, 'negative_ratio': 1.0, 'recency_weight_mean': 0.9865},
    {'company': '오로라', 'news_sentiment_score': np.float64(-90.47), 'news_count': 30, 'sentiment_volatility': 0.2622, 'positive_ratio': 0.0, 'negative_ratio': 0.9, 'recency_weight_mean': 0.9315},
    {'company': '오리콤', 'news_sentiment_score': np.float64(-58.07), 'news_count': 30, 'sentiment_volatility': 0.5689, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.458},
    {'company': '오텍', 'news_sentiment_score': np.float64(-31.61), 'news_count': 30, 'sentiment_volatility': 0.4774, 'positive_ratio': 0.0333, 'negative_ratio': 0.4, 'recency_weight_mean': 0.5738},
    {'company': '옴니시스템', 'news_sentiment_score': np.float64(-57.36), 'news_count': 30, 'sentiment_volatility': 0.5906, 'positive_ratio': 0.1, 'negative_ratio': 0.3, 'recency_weight_mean': 0.1729},
    {'company': '옵트론텍', 'news_sentiment_score': np.float64(-42.05), 'news_count': 30, 'sentiment_volatility': 0.4538, 'positive_ratio': 0.0, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.8716},
    {'company': '와이어블', 'news_sentiment_score': np.float64(-42.82), 'news_count': 30, 'sentiment_volatility': 0.4698, 'positive_ratio': 0.0, 'negative_ratio': 0.5, 'recency_weight_mean': 0.3306},
    {'company': '와이즈버즈', 'news_sentiment_score': np.float64(-59.91), 'news_count': 30, 'sentiment_volatility': 0.4413, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.8564},
    {'company': '와이지-원', 'news_sentiment_score': np.float64(-72.12), 'news_count': 30, 'sentiment_volatility': 0.5245, 'positive_ratio': 0.0333, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.6341},
    {'company': '우리기술투자', 'news_sentiment_score': np.float64(-70.28), 'news_count': 30, 'sentiment_volatility': 0.5661, 'positive_ratio': 0.0667, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.9126},
    {'company': '우성', 'news_sentiment_score': np.float64(-78.85), 'news_count': 30, 'sentiment_volatility': 0.3568, 'positive_ratio': 0.0, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9651},
    {'company': '우신시스템', 'news_sentiment_score': np.float64(-64.77), 'news_count': 30, 'sentiment_volatility': 0.4452, 'positive_ratio': 0.0, 'negative_ratio': 0.6333, 'recency_weight_mean': 0.6908},
    {'company': '우양에이치씨', 'news_sentiment_score': np.float64(-84.78), 'news_count': 30, 'sentiment_volatility': 0.606, 'positive_ratio': 0.1, 'negative_ratio': 0.4, 'recency_weight_mean': 0.2102},
    {'company': '우주일렉트로', 'news_sentiment_score': np.float64(-76.31), 'news_count': 30, 'sentiment_volatility': 0.5447, 'positive_ratio': 0.0333, 'negative_ratio': 0.6, 'recency_weight_mean': 0.342},
    {'company': '원림', 'news_sentiment_score': np.float64(-80.87), 'news_count': 27, 'sentiment_volatility': 0.4762, 'positive_ratio': 0.0, 'negative_ratio': 0.6296, 'recency_weight_mean': 0.498},
    {'company': '원익큐브', 'news_sentiment_score': np.float64(-39.68), 'news_count': 30, 'sentiment_volatility': 0.4788, 'positive_ratio': 0.0, 'negative_ratio': 0.4, 'recency_weight_mean': 0.3414},
    {'company': '월덱스', 'news_sentiment_score': np.float64(-34.23), 'news_count': 30, 'sentiment_volatility': 0.5289, 'positive_ratio': 0.0333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.2946},
    {'company': '웰크론', 'news_sentiment_score': np.float64(-53.41), 'news_count': 30, 'sentiment_volatility': 0.4546, 'positive_ratio': 0.0, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.4351},
    {'company': '유니온머티리얼', 'news_sentiment_score': np.float64(-1.85), 'news_count': 30, 'sentiment_volatility': 0.3818, 'positive_ratio': 0.0667, 'negative_ratio': 0.1333, 'recency_weight_mean': 0.6583},
    {'company': '유니켐', 'news_sentiment_score': np.float64(-52.78), 'news_count': 29, 'sentiment_volatility': 0.4614, 'positive_ratio': 0.0, 'negative_ratio': 0.4828, 'recency_weight_mean': 0.1427},
    {'company': '유니테스트', 'news_sentiment_score': np.float64(-43.8), 'news_count': 30, 'sentiment_volatility': 0.6304, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.7025},
    {'company': '유비온', 'news_sentiment_score': np.float64(-79.69), 'news_count': 30, 'sentiment_volatility': 0.4026, 'positive_ratio': 0.0, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.2372},
    {'company': '유비쿼스홀딩스', 'news_sentiment_score': np.float64(-52.88), 'news_count': 30, 'sentiment_volatility': 0.5552, 'positive_ratio': 0.0667, 'negative_ratio': 0.6, 'recency_weight_mean': 0.3701},
    {'company': '인스코비', 'news_sentiment_score': np.float64(-41.42), 'news_count': 30, 'sentiment_volatility': 0.5239, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.6891},
    {'company': '인지소프트', 'news_sentiment_score': np.float64(-24.76), 'news_count': 30, 'sentiment_volatility': 0.3985, 'positive_ratio': 0.0, 'negative_ratio': 0.2, 'recency_weight_mean': 0.1151},
    {'company': '인지컨트롤스', 'news_sentiment_score': np.float64(-37.67), 'news_count': 30, 'sentiment_volatility': 0.539, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.5996},
    {'company': '인텍플러스', 'news_sentiment_score': np.float64(-39.83), 'news_count': 30, 'sentiment_volatility': 0.566, 'positive_ratio': 0.1, 'negative_ratio': 0.5, 'recency_weight_mean': 0.9932}
]

# Create a DataFrame
df = pd.DataFrame(data)

# Ensure numeric columns are properly typed (especially after np.float64 conversion)
numeric_cols = ['news_sentiment_score', 'news_count', 'sentiment_volatility',
                'positive_ratio', 'negative_ratio', 'recency_weight_mean']
for col in numeric_cols:
    df[col] = df[col].astype(float)

# Save the DataFrame to a CSV file
csv_file_name = 'news_features.csv'
df.to_csv(csv_file_name, index=False)

print(f"DataFrame successfully created and saved as '{csv_file_name}'")

# Display the head of the created DataFrame for verification
print("\nDataFrame Head:")
print(df.head())

DataFrame successfully created and saved as 'news_features.csv'

DataFrame Head:
  company  news_sentiment_score  news_count  sentiment_volatility  \
0      CJ                -52.92        30.0                0.4649   
1  CJ ENM                -42.92        30.0                0.5189   
2  CJ제일제당                -13.13        30.0                0.5684   
3     HMM                -76.41        30.0                0.5532   
4      KT                -38.93        29.0                0.8260   

   positive_ratio  negative_ratio  recency_weight_mean  
0          0.0000          0.5333               0.9921  
1          0.0333          0.4667               0.8867  
2          0.1333          0.2667               0.9185  
3          0.0667          0.8333               0.9906  
4          0.2414          0.6207               0.9870  


In [3]:
import pandas as pd
import numpy as np
import os

# 제공된 데이터
new_data = [
    {'company': '일신바이오', 'news_sentiment_score': np.float64(-84.9), 'news_count': 30, 'sentiment_volatility': 0.6002, 'positive_ratio': 0.0667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.2611},
    {'company': '일진전기', 'news_sentiment_score': np.float64(-46.42), 'news_count': 30, 'sentiment_volatility': 0.5713, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.8445},
    {'company': '제이스코홀딩스', 'news_sentiment_score': np.float64(-55.94), 'news_count': 29, 'sentiment_volatility': 0.4737, 'positive_ratio': 0.0, 'negative_ratio': 0.5517, 'recency_weight_mean': 0.3177},
    {'company': '제이엠아이', 'news_sentiment_score': np.float64(-18.37), 'news_count': 30, 'sentiment_volatility': 0.6258, 'positive_ratio': 0.1333, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.1332},
    {'company': '조광페인트', 'news_sentiment_score': np.float64(-83.54), 'news_count': 30, 'sentiment_volatility': 0.4178, 'positive_ratio': 0.0333, 'negative_ratio': 0.9, 'recency_weight_mean': 0.7819},
    {'company': '조이시티', 'news_sentiment_score': np.float64(-0.49), 'news_count': 29, 'sentiment_volatility': 0.214, 'positive_ratio': 0.0345, 'negative_ratio': 0.0, 'recency_weight_mean': 0.9499},
    {'company': '조일알미늄', 'news_sentiment_score': np.float64(-49.58), 'news_count': 30, 'sentiment_volatility': 0.5531, 'positive_ratio': 0.0667, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.4964},
    {'company': '진로발효', 'news_sentiment_score': np.float64(-98.19), 'news_count': 30, 'sentiment_volatility': 0.1952, 'positive_ratio': 0.0, 'negative_ratio': 0.9667, 'recency_weight_mean': 0.0624},
    {'company': '진바이오텍', 'news_sentiment_score': np.float64(-68.89), 'news_count': 30, 'sentiment_volatility': 0.5341, 'positive_ratio': 0.0333, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.1978},
    {'company': '진성티이씨', 'news_sentiment_score': np.float64(-67.77), 'news_count': 30, 'sentiment_volatility': 0.4667, 'positive_ratio': 0.0, 'negative_ratio': 0.6667, 'recency_weight_mean': 0.4659},
    {'company': '진양산업', 'news_sentiment_score': np.float64(-43.28), 'news_count': 30, 'sentiment_volatility': 0.7362, 'positive_ratio': 0.2, 'negative_ratio': 0.4667, 'recency_weight_mean': 0.2255},
    {'company': '진양폴리', 'news_sentiment_score': np.float64(-64.19), 'news_count': 30, 'sentiment_volatility': 0.746, 'positive_ratio': 0.2, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.2199},
    {'company': '진양홀딩스', 'news_sentiment_score': np.float64(10.88), 'news_count': 30, 'sentiment_volatility': 0.5753, 'positive_ratio': 0.0667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.1646},
    {'company': '참엔지니어링', 'news_sentiment_score': np.float64(-23.69), 'news_count': 30, 'sentiment_volatility': 0.5903, 'positive_ratio': 0.1, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.1891},
    {'company': '체시스', 'news_sentiment_score': np.float64(-61.17), 'news_count': 30, 'sentiment_volatility': 0.5275, 'positive_ratio': 0.0333, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.3213},
    {'company': '카카오', 'news_sentiment_score': np.float64(-86.8), 'news_count': 30, 'sentiment_volatility': 0.4801, 'positive_ratio': 0.0667, 'negative_ratio': 0.9333, 'recency_weight_mean': 0.9985},
    {'company': '카페24', 'news_sentiment_score': np.float64(-32.0), 'news_count': 30, 'sentiment_volatility': 0.4172, 'positive_ratio': 0.0, 'negative_ratio': 0.3, 'recency_weight_mean': 0.961},
    {'company': '켐트로닉스', 'news_sentiment_score': np.float64(-20.91), 'news_count': 30, 'sentiment_volatility': 0.6896, 'positive_ratio': 0.1667, 'negative_ratio': 0.4333, 'recency_weight_mean': 0.5817},
    {'company': '코세스', 'news_sentiment_score': np.float64(-27.95), 'news_count': 30, 'sentiment_volatility': 0.5537, 'positive_ratio': 0.0667, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.6792},
    {'company': '코스맥스비티아이', 'news_sentiment_score': np.float64(-33.58), 'news_count': 30, 'sentiment_volatility': 0.5025, 'positive_ratio': 0.0333, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.4544},
    {'company': '코스모신소재', 'news_sentiment_score': np.float64(-32.49), 'news_count': 30, 'sentiment_volatility': 0.5035, 'positive_ratio': 0.0333, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.8191},
    {'company': '코오롱글로벌', 'news_sentiment_score': np.float64(-69.2), 'news_count': 30, 'sentiment_volatility': 0.5427, 'positive_ratio': 0.0667, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9469},
    {'company': '코칩', 'news_sentiment_score': np.float64(-27.6), 'news_count': 30, 'sentiment_volatility': 0.3504, 'positive_ratio': 0.0, 'negative_ratio': 0.1667, 'recency_weight_mean': 0.2941},
    {'company': '큐로홀딩스', 'news_sentiment_score': np.float64(-45.91), 'news_count': 29, 'sentiment_volatility': 0.4898, 'positive_ratio': 0.0345, 'negative_ratio': 0.5172, 'recency_weight_mean': 0.711},
    {'company': '큐에스아이', 'news_sentiment_score': np.float64(-24.94), 'news_count': 30, 'sentiment_volatility': 0.5779, 'positive_ratio': 0.0667, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.4276},
    {'company': '클래시스', 'news_sentiment_score': np.float64(-51.73), 'news_count': 29, 'sentiment_volatility': 0.6449, 'positive_ratio': 0.1034, 'negative_ratio': 0.5862, 'recency_weight_mean': 0.95},
    {'company': '태경케미컬', 'news_sentiment_score': np.float64(-57.25), 'news_count': 30, 'sentiment_volatility': 0.5961, 'positive_ratio': 0.0667, 'negative_ratio': 0.5, 'recency_weight_mean': 0.2403},
    {'company': '태광', 'news_sentiment_score': np.float64(-65.06), 'news_count': 30, 'sentiment_volatility': 0.6285, 'positive_ratio': 0.1, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.9838},
    {'company': '태웅', 'news_sentiment_score': np.float64(-57.07), 'news_count': 30, 'sentiment_volatility': 0.4769, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.7601},
    {'company': '테크엘', 'news_sentiment_score': np.float64(-14.43), 'news_count': 30, 'sentiment_volatility': 0.4057, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.012},
    {'company': '티케이지애강', 'news_sentiment_score': np.float64(-60.72), 'news_count': 30, 'sentiment_volatility': 0.5761, 'positive_ratio': 0.0667, 'negative_ratio': 0.7667, 'recency_weight_mean': 0.172},
    {'company': '티플랙스', 'news_sentiment_score': np.float64(-49.64), 'news_count': 30, 'sentiment_volatility': 0.5055, 'positive_ratio': 0.0333, 'negative_ratio': 0.7, 'recency_weight_mean': 0.3997},
    {'company': '퍼스텍', 'news_sentiment_score': np.float64(-33.54), 'news_count': 30, 'sentiment_volatility': 0.6171, 'positive_ratio': 0.1333, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.7272},
    {'company': '푸드웰', 'news_sentiment_score': np.float64(-19.06), 'news_count': 29, 'sentiment_volatility': 0.4993, 'positive_ratio': 0.0345, 'negative_ratio': 0.2759, 'recency_weight_mean': 0.2678},
    {'company': '풀무원', 'news_sentiment_score': np.float64(-25.09), 'news_count': 29, 'sentiment_volatility': 0.4432, 'positive_ratio': 0.0345, 'negative_ratio': 0.2414, 'recency_weight_mean': 0.9719},
    {'company': '프럼파스트', 'news_sentiment_score': np.float64(-77.07), 'news_count': 30, 'sentiment_volatility': 0.5506, 'positive_ratio': 0.0333, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.3526},
    {'company': '플레이위드', 'news_sentiment_score': np.float64(-14.14), 'news_count': 30, 'sentiment_volatility': 0.7546, 'positive_ratio': 0.2333, 'negative_ratio': 0.3667, 'recency_weight_mean': 0.8662},
    {'company': '하츠', 'news_sentiment_score': np.float64(-92.59), 'news_count': 30, 'sentiment_volatility': 0.2486, 'positive_ratio': 0.0, 'negative_ratio': 0.9333, 'recency_weight_mean': 0.9714},
    {'company': '한국공항', 'news_sentiment_score': np.float64(-85.49), 'news_count': 30, 'sentiment_volatility': 0.3855, 'positive_ratio': 0.0333, 'negative_ratio': 0.9333, 'recency_weight_mean': 0.9713},
    {'company': '한국알콜', 'news_sentiment_score': np.float64(-70.28), 'news_count': 29, 'sentiment_volatility': 0.52, 'positive_ratio': 0.0345, 'negative_ratio': 0.6897, 'recency_weight_mean': 0.279},
    {'company': '한국전력', 'news_sentiment_score': np.float64(-38.7), 'news_count': 30, 'sentiment_volatility': 0.561, 'positive_ratio': 0.0667, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.9842},
    {'company': '한농화성', 'news_sentiment_score': np.float64(-51.24), 'news_count': 30, 'sentiment_volatility': 0.5963, 'positive_ratio': 0.0667, 'negative_ratio': 0.5667, 'recency_weight_mean': 0.7507},
    {'company': '한빛소프트', 'news_sentiment_score': np.float64(-64.88), 'news_count': 30, 'sentiment_volatility': 0.5006, 'positive_ratio': 0.0333, 'negative_ratio': 0.7333, 'recency_weight_mean': 0.8782},
    {'company': '한성크린텍', 'news_sentiment_score': np.float64(-72.56), 'news_count': 28, 'sentiment_volatility': 0.462, 'positive_ratio': 0.0, 'negative_ratio': 0.4643, 'recency_weight_mean': 0.1802},
    {'company': '한신공영', 'news_sentiment_score': np.float64(-77.91), 'news_count': 30, 'sentiment_volatility': 0.3751, 'positive_ratio': 0.0, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9077},
    {'company': '한온시스템', 'news_sentiment_score': np.float64(-16.3), 'news_count': 30, 'sentiment_volatility': 0.3643, 'positive_ratio': 0.0, 'negative_ratio': 0.1667, 'recency_weight_mean': 0.9598},
    {'company': '한올바이오파마', 'news_sentiment_score': np.float64(-38.85), 'news_count': 30, 'sentiment_volatility': 0.5171, 'positive_ratio': 0.0333, 'negative_ratio': 0.3333, 'recency_weight_mean': 0.6415},
    {'company': '한창', 'news_sentiment_score': np.float64(-89.86), 'news_count': 26, 'sentiment_volatility': 0.2292, 'positive_ratio': 0.0385, 'negative_ratio': 0.9615, 'recency_weight_mean': 0.9517},
    {'company': '한화', 'news_sentiment_score': np.float64(-55.8), 'news_count': 30, 'sentiment_volatility': 0.5457, 'positive_ratio': 0.0333, 'negative_ratio': 0.6, 'recency_weight_mean': 0.9988},
    {'company': '한화에어로스페이스', 'news_sentiment_score': np.float64(-86.44), 'news_count': 30, 'sentiment_volatility': 0.3335, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.9668},
    {'company': '해성산업', 'news_sentiment_score': np.float64(-55.86), 'news_count': 30, 'sentiment_volatility': 0.7791, 'positive_ratio': 0.2, 'negative_ratio': 0.6, 'recency_weight_mean': 0.2210},
    {'company': '현대모비스', 'news_sentiment_score': np.float64(-1.29), 'news_count': 30, 'sentiment_volatility': 0.2491, 'positive_ratio': 0.0333, 'negative_ratio': 0.0333, 'recency_weight_mean': 0.9773},
    {'company': '현대지에프홀딩스', 'news_sentiment_score': np.float64(-86.83), 'news_count': 30, 'sentiment_volatility': 0.3155, 'positive_ratio': 0.0, 'negative_ratio': 0.8667, 'recency_weight_mean': 0.8559},
    {'company': '현대차', 'news_sentiment_score': np.float64(-74.53), 'news_count': 30, 'sentiment_volatility': 0.5402, 'positive_ratio': 0.0667, 'negative_ratio': 0.8, 'recency_weight_mean': 0.9977},
    {'company': '형지I&C', 'news_sentiment_score': np.float64(-78.7), 'news_count': 30, 'sentiment_volatility': 0.3705, 'positive_ratio': 0.0, 'negative_ratio': 0.8, 'recency_weight_mean': 0.8969},
    {'company': '형지엘리트', 'news_sentiment_score': np.float64(-31.02), 'news_count': 30, 'sentiment_volatility': 0.4288, 'positive_ratio': 0.0, 'negative_ratio': 0.2667, 'recency_weight_mean': 0.9071},
    {'company': '호텔신라', 'news_sentiment_score': np.float64(-92.55), 'news_count': 26, 'sentiment_volatility': 0.2096, 'positive_ratio': 0.0, 'negative_ratio': 0.9615, 'recency_weight_mean': 0.9695},
    {'company': '화성밸브', 'news_sentiment_score': np.float64(-62.6), 'news_count': 30, 'sentiment_volatility': 0.4756, 'positive_ratio': 0.0, 'negative_ratio': 0.5333, 'recency_weight_mean': 0.2798},
    {'company': '화승코퍼레이션', 'news_sentiment_score': np.float64(-74.31), 'news_count': 30, 'sentiment_volatility': 0.3862, 'positive_ratio': 0.0, 'negative_ratio': 0.8, 'recency_weight_mean': 0.3597},
    {'company': '효성', 'news_sentiment_score': np.float64(-59.62), 'news_count': 30, 'sentiment_volatility': 0.4846, 'positive_ratio': 0.0, 'negative_ratio': 0.6, 'recency_weight_mean': 0.9781},
    {'company': '효성ITX', 'news_sentiment_score': np.float64(-31.3), 'news_count': 29, 'sentiment_volatility': 0.781, 'positive_ratio': 0.2759, 'negative_ratio': 0.4138, 'recency_weight_mean': 0.2999}
]

file_name = 'news_features.csv'

# float64 타입을 일반 float으로 변환 (CSV 저장을 위해)
for item in new_data:
    for key, value in item.items():
        if isinstance(value, np.float64):
            item[key] = float(value)

# DataFrame 생성
df_new = pd.DataFrame(new_data)

# 'company' 열을 첫 번째 열로 재배치 (기존 파일 형식과 일치하도록 보장)
column_order = ['company', 'news_sentiment_score', 'news_count', 'sentiment_volatility', 'positive_ratio', 'negative_ratio', 'recency_weight_mean']
df_new = df_new[column_order]

# 파일 추가 (Append) 로직
if os.path.exists(file_name):
    # 파일이 존재하면: 헤더 없이, 추가 모드('a')로 데이터만 추가
    print(f"'{file_name}' 파일이 존재합니다. 새 데이터를 추가합니다.")
    df_new.to_csv(
        file_name,
        mode='a',          # 추가 모드 (Append mode)
        header=False,      # 헤더를 쓰지 않음
        index=False,
        encoding='utf-8-sig'
    )
    print("데이터가 성공적으로 추가되었습니다.")
else:
    # 파일이 존재하지 않으면: 새 파일을 생성하고 헤더와 함께 쓰기
    print(f"'{file_name}' 파일이 존재하지 않습니다. 새 파일을 생성합니다.")
    df_new.to_csv(
        file_name,
        mode='w',          # 쓰기 모드 (Write mode)
        header=True,       # 헤더를 씀
        index=False,
        encoding='utf-8-sig'
    )
    print("새 파일에 데이터가 저장되었습니다.")

# (선택 사항) 추가된 데이터 확인
try:
    df_full = pd.read_csv(file_name, encoding='utf-8-sig')
    print("\n최종 데이터프레임의 마지막 5행:")
    print(df_full.tail())
    print(f"\n총 데이터 개수: {len(df_full)}")
except Exception as e:
    print(f"\n파일 읽기 중 오류 발생: {e}")

'news_features.csv' 파일이 존재합니다. 새 데이터를 추가합니다.
데이터가 성공적으로 추가되었습니다.

최종 데이터프레임의 마지막 5행:
     company  news_sentiment_score  news_count  sentiment_volatility  \
198     호텔신라                -92.55        26.0                0.2096   
199     화성밸브                -62.60        30.0                0.4756   
200  화승코퍼레이션                -74.31        30.0                0.3862   
201       효성                -59.62        30.0                0.4846   
202    효성ITX                -31.30        29.0                0.7810   

     positive_ratio  negative_ratio  recency_weight_mean  
198          0.0000          0.9615               0.9695  
199          0.0000          0.5333               0.2798  
200          0.0000          0.8000               0.3597  
201          0.0000          0.6000               0.9781  
202          0.2759          0.4138               0.2999  

총 데이터 개수: 203
